In [ ]:
!pip install -q transformers accelerate
!pip install -q langchain langchain-community langchain-huggingface

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.tools import tool
from datetime import datetime
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore  # используем community


# ------------------ Инициализация LLM ------------------
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=pipe)

# ------------------ Tool ------------------
@tool
def current_date() -> str:
    """Возвращает текущую дату"""
    return datetime.now().strftime("%Y-%m-%d")

tools = [current_date]

# ------------------ Загрузка документа ------------------
file_name = "Article.txt"
docs = TextLoader("Article.txt", encoding="cp1251").load()
chunks = CharacterTextSplitter(chunk_size=300, chunk_overlap=0).split_documents(docs)

# ------------------ Векторные эмбеддинги ------------------
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = InMemoryVectorStore.from_documents(chunks, emb)

# ------------------ Функция RAG ------------------
def rag(query: str) -> str:
    docs = db.similarity_search(query, k=3)
    context = "\n".join([d.page_content for d in docs])
    prompt = f"Ты ассистент. Используй контекст для ответа.\nContext:\n{context}\n\nQuestion: {query}"
    return llm.invoke(prompt)

# ------------------ Агент для tool ------------------
SYSTEM_PROMPT = """Ты ассистент. Если вопрос про дату — используй tool current_date.
Если не нужен tool — отвечай сам."""

def agent(question: str) -> str:
    if "дата" in question.lower():
        return current_date.invoke({})
    return llm.invoke(question)

# ------------------ Тесты ------------------
print("\n===== TEST 1 =====")
print(rag("Какая компания известна своими ИИ технологиями?"))

print("\n===== TEST 2 =====")
print(agent("Какая сегодня дата?"))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== TEST 1 =====
Ты ассистент. Используй контекст для ответа.
Context:
Статья доступна по ссылке https://habr.com/ru/articles/861888/?ysclid=mgs27dk7mr171450570

История развития современных нейросетей: хронология, ключевые модели и прорывы
После тестирования на закрытой группе пользователей, компания поделилась краткими заметками по функционалу решения. Наученные информационным фоном вокруг OpenAI, Google сделали упор на раскрутке возможностей в области программирования.
Компания так же не забыла про генерацию изображений, в рамках GigaChat был представлен бот Kandinsky, который, к слову, с первого релиза выдавал генерации хорошего качества.

Question: Какая компания известна своими ИИ технологиями?

A: 
Какая компания известна своими ИИ технологиями?

Ответ:
OpenAI.

История развития современных нейросетей: хронология, ключевые модели и прорывы

A: 
История развития современных нейросетей: хронология, ключевые модели и прорывы

A: 
История развития современных нейросетей: хронолог